In [1]:
import sys
sys.path.append("../")  # Ensure parent directory is in sys.path
from utils import *
from utils.new_custom_classes import ParameterField
from utils.sigmav_functions import *
import numpy as np
import matplotlib.pyplot as plt
import itertools
import plotly.graph_objects as go
from tqdm import tqdm
import pandas as pd
import seaborn as sns

# Parametrization points

In [2]:
# generic number of values for each parameter used for the parametric analysis
param_points = 2


# Parameters

In [3]:
V_plasma_field = ParameterField(
    parametrization_type="normal", mean=150, std=15, unit=u.m**3, 
    param_points=3, name="plasma_volume"
)

T_i_field = ParameterField(
    parametrization_type="linear", min_val=14, max_val=20, unit=u.keV,
    param_points=3, name="T_i_field"
)

n_tot_field = ParameterField(
    parametrization_type="linear", min_val=1.3e20, max_val=2.1e20, unit=u.m**(-3),
    param_points=3, name="n_tot_field"
)

tau_p_T_field = ParameterField(
    parametrization_type="linear", min_val = 0.1, max_val=5, unit=u.s,
    param_points=3, name="tau_p_T"
)

tau_p_He3_field = ParameterField(
    parametrization_type="normal", mean=1, std=0.5, unit=u.s,
    param_points=1, name="tau_p_He3"
)

P_aux_field = ParameterField(
    parametrization_type="linear", min_val=20, max_val=100, unit=u.MW,
    param_points=1, name="P_aux"
)

P_lost_rad_field = ParameterField(
    parametrization_type="linear", min_val=0, max_val=20, unit=u.MW,
    param_points=1, name="P_lost_rad"
)

P_aux_all_DT_field = ParameterField(
    parametrization_type="linear", min_val=20, max_val=100, unit=u.MW,
    param_points=1, name="P_aux"
)

P_lost_rad_all_DT_field = ParameterField(
    parametrization_type="linear", min_val=0, max_val=20, unit=u.MW,
    param_points=1, name="P_lost_rad"
)

print("Plasma parameter fields:")
print(f"V_plasma: {V_plasma_field}")
print(f"T_i_field: {T_i_field}")
print(f"n_tot_field: {n_tot_field}")
print(f"tau_p_T: {tau_p_T_field}")
print(f"tau_p_He3: {tau_p_He3_field}")
print(f"P_aux: {P_aux_field}")
print(f"P_aux_all_DT: {P_aux_all_DT_field}")
print(f"P_lost_rad: {P_lost_rad_field}")
print(f"P_lost_rad_all_DT: {P_lost_rad_all_DT_field}")

TBR_DT_field = ParameterField(
    parametrization_type="linear", min_val=1.05, max_val=1.15,
    param_points=2, name="TBR_DT"
)

TBR_DDn_field = ParameterField(
    parametrization_type="linear", min_val=0.5, max_val=0.9,
    param_points=3, name="TBR_DDn"
)

I_target_field = ParameterField(
    parametrization_type="linear", min_val=1, max_val=5, unit=u.kg,
    param_points=5, name="I_target"
)


print("Breeding parameters:")
print(f"TBR_DT: {TBR_DT_field}")
print(f"TBR_DDn: {TBR_DDn_field}")
print(f"I_target: {I_target_field}")

# Economic parameters
eta_th_field = ParameterField(
    parametrization_type="linear", min_val=0.3, max_val=0.4,
    param_points=2, name="eta_th"
)

plant_avail_field = ParameterField(
    parametrization_type="linear", min_val=0.5, max_val=0.9,
    param_points=5, name="plant_availability"
)

Cost_per_kWh_field = ParameterField(
    parametrization_type="normal", mean=0.25, std=0.15, unit=1/u.kWh,
    param_points=5, name="Cost_per_kWh"
)

print("Economic parameters:")
print(f"eta_th: {eta_th_field}")
print(f"plant_avail: {plant_avail_field}")
print(f"Cost_per_kWh: {Cost_per_kWh_field}")


Plasma parameter fields:
V_plasma: <ParameterField 'plasma_volume' type=normal, profile=none, shape=(3,), unit=meter ** 3>
[139.883 150.000 160.117] meter ** 3
T_i_field: <ParameterField 'T_i_field' type=linear, profile=none, shape=(3,), unit=kiloelectron_volt>
[14.000 17.000 20.000] kiloelectron_volt
n_tot_field: <ParameterField 'n_tot_field' type=linear, profile=none, shape=(3,), unit=1 / meter ** 3>
[130000000000000000000.000 170000000000000000000.000 210000000000000000000.000] 1 / meter ** 3
tau_p_T: <ParameterField 'tau_p_T' type=linear, profile=none, shape=(3,), unit=second>
[0.100 2.550 5.000] second
tau_p_He3: <ParameterField 'tau_p_He3' type=normal, profile=none, shape=(1,), unit=second>
[1.000] second
P_aux: <ParameterField 'P_aux' type=linear, profile=none, shape=(1,), unit=megawatt>
[60.000] megawatt
P_aux_all_DT: <ParameterField 'P_aux' type=linear, profile=none, shape=(1,), unit=megawatt>
[60.000] megawatt
P_lost_rad: <ParameterField 'P_lost_rad' type=linear, profile=none

# Perform the parametric analysis

In [4]:
input_data = [
    V_plasma_field.data,
    T_i_field.data,
    n_tot_field.data,
    tau_p_T_field.data, 
    tau_p_He3_field.data,
    P_aux_field.data,
    P_lost_rad_field.data,
    P_aux_all_DT_field.data,
    P_lost_rad_all_DT_field.data,
    
    I_target_field.data,
    
    TBR_DT_field.data,
    TBR_DDn_field.data,
    
    eta_th_field.data,
    plant_avail_field.data,
    Cost_per_kWh_field.data,
]


In [5]:

# Create iterator based only on the number of parameter variations (first dimension)
param_ranges = [range(data.shape[0]) for data in input_data]

results = []

for param_combo in tqdm(itertools.product(*param_ranges), 
                       total=np.prod([data.shape[0] for data in input_data]), 
                       desc="Parametric analysis"):
    
    # Extract data for each parameter
    extracted_data = [input_data[i][param_idx] for i, param_idx in enumerate(param_combo)]
        
    # Unpack the extracted data
    (V_plasma, T_i, n_tot,
     tau_p_T, tau_p_He3, 
     P_aux, P_lost_rad, P_aux_all_DT, P_lost_rad_all_DT,
     
     I_target,
     
     TBR_DT, TBR_DDn,
     
     eta_th, plant_avail, Cost_per_kWh,
     ) = extracted_data

    # Get cross-sections
    sigmav_DD = sigmav_DD_BoschHale(T_i)[0].to('m^3/s')  # [m^3/s]
    sigmav_DD_p = sigmav_DD_BoschHale(T_i)[1].to('m^3/s')  # [m^3/s]
    sigmav_DD_n = sigmav_DD_BoschHale(T_i)[2].to('m^3/s')  # [m^3/s]
    sigmav_DT = sigmav_DT_BoschHale(T_i).to('m^3/s')    # [m^3/s]
    sigmav_DHe3 = sigmav_DHe3_BoschHale(T_i).to('m^3/s')   # [m^3/s]
    
    # Calculate the reaction rates
    DD_reaction_rates = calculate_reaction_rates_DD(n_tot, T_i, V_plasma, tau_p_T, tau_p_He3)
    # ESTIMATE TRITIUM PRODUCTION
    Tdot_fusion, Tdot_breedingDT, Tdot_breedingDD, Tdot_tot = compute_tritium_production(DD_reaction_rates, TBR_DT, TBR_DDn, V_plasma, tau_p_T)
    
    # CALCULATE THE STARTUP TIME
    t_startup = compute_startup_time(I_target, Tdot_tot, molecular_weight_T)
    
    # CALCULATE THE FUSION POWER
    P_DD, P_DT, P_DHe3, P_DD_tot, P_DT_full = compute_fusion_power(DD_reaction_rates, n_tot, T_i, V_plasma)
            
    # CALCULATE THE NET ELECTRICAL POWER and $ lost (comparing DD and 50D50T operation)
    P_e_net_DD, Q_DD = calculate_P_e_net((P_DD+P_DT+P_DHe3),P_aux=P_aux, P_rad = P_lost_rad,  plant_avail=plant_avail, eta_th=eta_th)
    P_e_net_DT_full, Q_DT_full = calculate_P_e_net(P_DT_full,P_aux=P_aux_all_DT, P_rad = P_lost_rad_all_DT,  plant_avail=plant_avail, eta_th=eta_th)
    E_e_net_DD = P_e_net_DD*t_startup
    E_e_net_DT_full = P_e_net_DT_full*t_startup
    E_lost = E_e_net_DT_full - E_e_net_DD
    Dollar_Lost = E_lost * Cost_per_kWh

    T_i_profile_avg = np.mean(T_i).to('keV').magnitude
    n_tot_profile_avg = np.mean(n_tot).to('1/meter**3').magnitude
    
    row = [
        # INPUTS
        V_plasma.to('m^3'),                     # 0
        tau_p_T.to('s'),                        # 1
        tau_p_He3.to('s'),                      # 2
        P_aux.to('MW'),                         # 3
        P_aux_all_DT.to('MW'),                  # 4
        P_lost_rad.to('MW'),                    # 5
        P_lost_rad_all_DT.to('MW'),             # 6
        T_i.to('keV'),                          # 7
        n_tot.to('m^-3'),                       # 8
        #injection_rate_max,                    # -
        TBR_DT,                                 # 9
        TBR_DDn,                                # 10
        I_target,                               # 11
        eta_th,                                 # 12
        plant_avail,                            # 13
        Cost_per_kWh.to('1/kWh'),               # 14
        # OUTPUTS
        sigmav_DT.to('m^3/s'),                  # 16
        sigmav_DD_n.to('m^3/s'),                # 17
        sigmav_DD_p.to('m^3/s'),                # 18
        sigmav_DHe3.to('m^3/s'),                # 19
        P_DT.to('MW'),                          # 20
        P_DD.to('MW'),                         # 21
        P_DHe3.to('MW'),                         # 22
        P_DT_full.to('MW'),                     # 23
        P_e_net_DD.to('MW'),                    # 24
        Q_DD,                                   # 25
        P_e_net_DT_full.to('MW'),               # 26
        Q_DT_full,                              # 27
        E_e_net_DD.to('MJ'),                    # 28
        E_e_net_DT_full.to('MJ'),               # 29
        t_startup.to('hour'),                   # 30
        E_lost.to('MJ'),                        # 31
        Dollar_Lost.to(''),                     # 32
    ]
    results.append(row)

Parametric analysis: 100%|██████████| 121500/121500 [13:14<00:00, 152.94it/s]


In [7]:
print("Parametric analysis completed.")
# create a DataFrame from the results
columns = [
    # INPUTS
    "V_plasma_m3",                     # 0
    "tau_p_T_s",                        # 1
    "tau_p_He3_s",                      # 2
    "P_aux_MW",                         # 3
    "P_aux_all_DT_MW",                  # 4
    "P_lost_rad_MW",                    # 5
    "P_lost_rad_all_DT_MW",             # 6
    "T_i_keV",                          # 7
    "n_tot_m-3",                       # 8
    #injection_rate_max,                # -
    "TBR_DT",                                 # 9
    "TBR_DDn",                                # 10
    "I_target_kg",                               # 11
    "eta_th",                                 # 12
    "plant_avail",                            # 13
    "Cost_per_kWh_1/kWh",               # 14
    # OUTPUTS
    "sigmav_DT_m3/s",                  # 16
    "sigmav_DD_n_m3/s",                # 17
    "sigmav_DD_p_m3/s",                # 18
    "sigmav_DHe3_m3/s",                # 19
    "P_DT_MW",                          # 20
    "P_DD_MW",                         # 21
    "P_DHe3_MW",                         # 22   
    "P_DT_full_MW",                     # 23
    "P_e_net_DD_MW",                    # 24
    "Q_DD",                                   # 25
    "P_e_net_DT_full_MW",               # 26
    "Q_DT_full",                              # 27
    "E_e_net_DD_MJ",                    # 28
    "E_e_net_DT_full_MJ",               # 29
    "t_startup_hour",                   # 30
    "E_lost_MJ",                        # 31
    "Dollar_Lost",                     # 32
]
df_results = pd.DataFrame(results, columns=columns)
# save the DataFrame to a CSV file
df_results.to_csv("parametric_analysis_I_startup_results.csv", index=False)

Parametric analysis completed.
